In [0]:
%restart_python

In [0]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_URL = os.getenv("ERP_API_URL")
TOKEN = os.getenv("ERP_API_TOKEN")

def fetch_all_pages(url: str, token: str) -> list:
    headers = {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    all_records = []
    page = 1

    while url:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        all_records.extend(data.get("value", []))
        print(f"Page {page}: {len(data.get('value', []))} records (total so far: {len(all_records)})")
        url = data.get("@odata.nextLink")
        page += 1

    return all_records

records = fetch_all_pages(API_URL, TOKEN)
df = pd.DataFrame(records)
print("Final shape:", df.shape)

In [0]:
print(df.shape)
print(df.dtypes)
print(df.columns.tolist())

In [0]:
df["postingDate"] = pd.to_datetime(df["postingDate"])
print("Date range:", df["postingDate"].min(), "to", df["postingDate"].max())
print("Total days spanned:", (df["postingDate"].max() - df["postingDate"].min()).days)

# Check for gaps — days with zero records at all
date_counts = df.groupby(df["postingDate"].dt.date).size()
full_range = pd.date_range(df["postingDate"].min(), df["postingDate"].max())
missing_days = set(full_range.date) - set(date_counts.index)
print(f"Days with zero entries: {len(missing_days)} out of {len(full_range)}")

In [0]:
print(df["entryType"].value_counts())
print(df["documentType"].value_counts())

In [0]:
print(df["quantity"].describe())
print("Negative qty count:", (df["quantity"] < 0).sum())
print("Positive qty count:", (df["quantity"] > 0).sum())
print("Zero qty count:", (df["quantity"] == 0).sum())

# Cross-check: is negative quantity always tied to Sales Shipment?
print(df.groupby("documentType")["quantity"].agg(["mean", "min", "max"]))

In [0]:
print("Unique items:", df["itemNo"].nunique())
print("Item categories:", df["itemCategoryCode"].unique())
print("Sub divisions:", df["subDivision"].unique())
print("Brand codes:", df["brandCode"].unique())
print("Location codes:", df["locationCode"].nunique())
print(df["locationCode"].value_counts().head(20))
print("Vehicle types:", df["itemCategory2"].unique())
print("Brand Description:", df["brandDescription"].unique())
print("Country Code:", df["countryRegionCode"].unique())

print("\nBrand code counts:")
print(df["brandCode"].value_counts())

In [0]:
print(df.isnull().sum().sort_values(ascending=False))

In [0]:
print(df[["quantity", "costAmountActual", "salesAmountActual"]].describe())